In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import math
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

TEST_CSV     = '/content/drive/MyDrive/video_koordinat_veriseti_TEST.csv'
MODEL_H5     = '/content/drive/MyDrive/isaret_dili_beyni_v2.h5'

IDX_NOSE_X  = 0;  IDX_NOSE_Y  = 1
IDX_RH_X    = 99; IDX_RH_Y    = 100
IDX_LH_X    = 162; IDX_LH_Y   = 163
IDX_MOUTH_X = 27; IDX_MOUTH_Y = 28
IDX_REYE_X  = 6;  IDX_REYE_Y  = 7
IDX_LEYE_X  = 15; IDX_LEYE_Y  = 16
IDX_RS_X    = 33; IDX_RS_Y    = 34
IDX_LS_X    = 36; IDX_LS_Y    = 37

def dist(x1, y1, x2, y2):
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def extract_features(raw_frame, prev_rh, prev_lh):
    nx = raw_frame[IDX_NOSE_X]
    ny = raw_frame[IDX_NOSE_Y]

    base = []
    # Pose (25), Sağ El (21), Sol El (21)
    for i in range(25):
        base.extend([(raw_frame[i*3]-nx)/2.0, (raw_frame[i*3+1]-ny)/2.0, raw_frame[i*3+2]/2.0])
    for i in range(21):
        base.extend([(raw_frame[99+i*3]-nx)/2.0, (raw_frame[99+i*3+1]-ny)/2.0, raw_frame[99+i*3+2]/2.0])
    for i in range(21):
        base.extend([(raw_frame[162+i*3]-nx)/2.0, (raw_frame[162+i*3+1]-ny)/2.0, raw_frame[162+i*3+2]/2.0])

    rhX, rhY = raw_frame[IDX_RH_X], raw_frame[IDX_RH_Y]
    lhX, lhY = raw_frame[IDX_LH_X], raw_frame[IDX_LH_Y]

    extra = [
        dist(rhX, rhY, raw_frame[IDX_MOUTH_X], raw_frame[IDX_MOUTH_Y]),
        dist(rhX, rhY, raw_frame[IDX_REYE_X],  raw_frame[IDX_REYE_Y]),
        dist(rhX, rhY, raw_frame[IDX_NOSE_X],  raw_frame[IDX_NOSE_Y]),
        dist(rhX, rhY, raw_frame[IDX_RS_X],    raw_frame[IDX_RS_Y]),
        dist(lhX, lhY, raw_frame[IDX_MOUTH_X], raw_frame[IDX_MOUTH_Y]),
        dist(lhX, lhY, raw_frame[IDX_LEYE_X],  raw_frame[IDX_LEYE_Y]),
        dist(lhX, lhY, raw_frame[IDX_NOSE_X],  raw_frame[IDX_NOSE_Y]),
        dist(lhX, lhY, raw_frame[IDX_LS_X],    raw_frame[IDX_LS_Y]),
        dist(rhX, rhY, prev_rh[0], prev_rh[1]),
        dist(lhX, lhY, prev_lh[0], prev_lh[1])
    ]
    return np.array(base + extra, dtype=np.float32), (rhX, rhY), (lhX, lhY)

df = pd.read_csv(TEST_CSV)
print(f"Ham veri shape: {df.shape}")

META_COLS  = ['video_name', 'class_id', 'frame_no']
COORD_COLS = [c for c in df.columns if c not in META_COLS]

TIMESTEPS = 60
FEATURES  = 211
sequences, labels = [], []

grouped = df.groupby(['video_name', 'class_id'], sort=False)
print(f"Toplam video (örnek) sayısı: {len(grouped)}")

for (video_name, class_id), group in grouped:
    group = group.sort_values('frame_no')
    raw_frames = group[COORD_COLS].values.astype(np.float32)
    n = len(raw_frames)

    # EĞİTİMDEKİ GİBİ: Baştan al, eksikse sona sıfır (Post-padding) ekle
    if n >= TIMESTEPS:
        selected_raw_frames = raw_frames[:TIMESTEPS]
    else:
        pad_size = TIMESTEPS - n
        pad_raw = np.zeros((pad_size, len(COORD_COLS)), dtype=np.float32)
        selected_raw_frames = np.vstack([raw_frames, pad_raw])

    processed_seq = []
    prev_rh, prev_lh = (0.0, 0.0), (0.0, 0.0)

    for raw_frame in selected_raw_frames:
        if np.all(raw_frame == 0):
            processed_seq.append(np.zeros(FEATURES, dtype=np.float32))
        else:
            feat, prev_rh, prev_lh = extract_features(raw_frame, prev_rh, prev_lh)
            processed_seq.append(feat)

    sequences.append(processed_seq)
    labels.append(class_id)

X = np.array(sequences, dtype=np.float32)
y_raw = np.array(labels)

if str(y_raw[0]).isdigit():
    y = y_raw.astype(int)
    class_names = np.unique(y)
else:
    le = LabelEncoder()
    y  = le.fit_transform(y_raw)
    class_names = le.classes_

NUM_CLASSES = len(class_names)
print(f"Sequence array shape : {X.shape}")
print(f"Sınıf sayısı         : {NUM_CLASSES}")


print("\n── H5 modeli yükleniyor ──")
model_h5 = tf.keras.models.load_model(MODEL_H5)

probs_h5  = model_h5.predict(X, batch_size=32, verbose=1)
y_pred_h5 = np.argmax(probs_h5, axis=1)

acc_h5       = accuracy_score(y, y_pred_h5) * 100
f1_macro     = f1_score(y, y_pred_h5, average='macro', zero_division=0) * 100
precision_m  = precision_score(y, y_pred_h5, average='macro', zero_division=0) * 100
recall_m     = recall_score(y, y_pred_h5, average='macro', zero_division=0) * 100

print("\n" + "="*50)
print("          TEST SONUÇ ÖZETİ (H5 MODELİ)")
print("="*50)
print(f" Video (örnek) sayısı : {len(y):,}")
print(f" Sınıf sayısı         : {NUM_CLASSES}")
print(f" Accuracy             : {acc_h5:.2f}%")
print(f" F1 (macro)           : {f1_macro:.2f}%")
print(f" Precision (macro)    : {precision_m:.2f}%")
print(f" Recall (macro)       : {recall_m:.2f}%")
print("="*50)

Ham veri shape: (166357, 228)
Toplam video (örnek) sayısı: 2808
Sequence array shape : (2808, 60, 211)
Sınıf sayısı         : 226

── H5 modeli yükleniyor ──


88/88 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step

          TEST SONUÇ ÖZETİ (H5 MODELİ)
 Video (örnek) sayısı : 2,808
 Sınıf sayısı         : 226
 Accuracy             : 78.42%
 F1 (macro)           : 78.00%
 Precision (macro)    : 79.97%
 Recall (macro)       : 78.36%
